# Toxicity Classification and Fairness in Online Moderation

## 01 - Data Preparation

This notebook prepares the Civil Comments dataset for
binary toxicity classification

Objectives:
1. Load and inspect the Civil Comments dataset
2. Identify missing values, empty comments, and duplicate text
3. Remove duplicate comments within and across dataset splits
4. Preserve independent training, validation, and testing sets
5. Convert continuous toxicity scores into binary classification labels
6. Verify the integrity and class distribution of the cleaned dataset

### Import necessary packages - load and inspect data

In [6]:
# imports and load data
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict

ds = load_dataset("google/civil_comments");

In [7]:
# binary classification threshold
TOXICITY_THRESHOLD = 0.5

# dataset splits
SPLITS = ["train", "validation", "test"]

# primary columns
TEXT_COLUMN = "text"
TARGET_COLUMN = "toxicity"
LABEL_COLUMN = "label"

In [8]:
# display dataset structure
print(ds)

# display available columns
print(ds["train"].column_names)

# display first observation
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 1804874
    })
    validation: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
    test: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 97320
    })
})
['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit']
{'text': "This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!", 'toxicity': 0.0, 'severe_toxicity': 0.0, 'obscene': 0.0, 'threat': 0.0, 'insult': 0.0, 'identity_attack': 0.0, 'sexual_explicit': 0.0}


### Data cleaning

In [10]:
# check for missing values, empty comments, and duplicate text
for split in SPLITS:
    df = ds[split].to_pandas()

    print(f"\n--- {split.upper()} ---")

    # missing values
    print("Missing text:", df[TEXT_COLUMN].isna().sum())
    print("Missing toxicity:", df[TARGET_COLUMN].isna().sum())

    # empty or whitespace-only comments
    empty = df[TEXT_COLUMN].str.strip().eq("").sum()
    print("Empty comments:", empty)

    # duplicate comments within this split
    duplicates = df[TEXT_COLUMN].duplicated().sum()
    print("Duplicate comments:", duplicates)


--- TRAIN ---
Missing text: 0
Missing toxicity: 0
Empty comments: 0
Duplicate comments: 24051

--- VALIDATION ---
Missing text: 0
Missing toxicity: 0
Empty comments: 0
Duplicate comments: 474

--- TEST ---
Missing text: 0
Missing toxicity: 0
Empty comments: 0
Duplicate comments: 458


#### Removing duplicates

Through our data exploration, we identified a substantial number of duplicate comments. Of the 1,804,874 rows in the training set, 34,231 contain text that appears more than once. Among these, 3,886 distinct groups of duplicate comments have inconsistent toxicity scores.

To address these duplicates and prevent potential data leakage, we removed repeated comments both within and across dataset splits. We prioritized retaining comments in the test set, followed by the validation set and then the training set, ensuring that no identical comments appear in multiple splits. Within each split, we retained only the first occurrence of each unique comment.

One limitation of this approach is that retaining the first occurrence arbitrarily selects a toxicity score when duplicate comments have conflicting annotations. While this method provides a simple and reproducible preprocessing strategy, it may discard meaningful differences in human toxicity evaluations.

In [12]:
# remove duplicate comments within each dataset split
# keep the first occurrence of each unique comment
train_df = (
    ds["train"]
    .to_pandas()
    .drop_duplicates(subset=["text"], keep="first")
)

val_df = (
    ds["validation"]
    .to_pandas()
    .drop_duplicates(subset=["text"], keep="first")
)

test_df = (
    ds["test"]
    .to_pandas()
    .drop_duplicates(subset=["text"], keep="first")
)

# display updated dataset sizes
print("Dataset sizes after removing internal duplicates:")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Dataset sizes after removing internal duplicates:
Train: 1780823
Validation: 96846
Test: 96862


In [13]:
# remove comments from validation that also appear in test
val_df = val_df[
    ~val_df["text"].isin(test_df["text"])
].copy()

# remove comments from training that appear in validation or test
train_df = train_df[
    ~train_df["text"].isin(val_df["text"])
    & ~train_df["text"].isin(test_df["text"])
].copy()

# reset DataFrame indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# display updated dataset sizes
print("Dataset sizes after removing overlapping comments:")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Dataset sizes after removing overlapping comments:
Train: 1778797
Validation: 96255
Test: 96862


In [14]:
# reconstruct cleaned dataset
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

# display cleaned dataset
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 1778797
    })
    validation: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 96255
    })
    test: Dataset({
        features: ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit'],
        num_rows: 96862
    })
})


In [15]:
# check for duplicates within each split
for split in ["train", "validation", "test"]:
    df = ds[split].to_pandas()

    print(f"\n--- {split.upper()} ---")
    print("Total comments:", len(df))
    print("Duplicate comments:", df["text"].duplicated().sum())

    print(
        "Percent toxic:",
        round((df["toxicity"] >= 0.5).mean() * 100, 2)
    )

# check for overlapping comments across splits
train_texts = set(ds["train"]["text"])
val_texts = set(ds["validation"]["text"])
test_texts = set(ds["test"]["text"])

print("\n--- CROSS-SPLIT OVERLAP ---")

print("Train / Validation:", len(train_texts & val_texts))
print("Train / Test:", len(train_texts & test_texts))
print("Validation / Test:", len(val_texts & test_texts))


--- TRAIN ---
Total comments: 1778797
Duplicate comments: 0
Percent toxic: 8.0

--- VALIDATION ---
Total comments: 96255
Duplicate comments: 0
Percent toxic: 7.88

--- TEST ---
Total comments: 96862
Duplicate comments: 0
Percent toxic: 7.99

--- CROSS-SPLIT OVERLAP ---
Train / Validation: 0
Train / Test: 0
Validation / Test: 0


#### Verify toxicity scores in expected range

In [17]:
# verify toxicity scores are within the expected range
for split in SPLITS:
    scores = np.array(ds[split][TARGET_COLUMN])

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    print(f"{split}: toxicity scores valid")

train: toxicity scores valid
validation: toxicity scores valid
test: toxicity scores valid


### Create binary label column

In [19]:
# create binary label
def add_binary_label(example):
    example[LABEL_COLUMN] = int(
        example[TARGET_COLUMN] >= TOXICITY_THRESHOLD
    )
    return example

# apply to training, validation, and testing sets
ds = ds.map(add_binary_label);

Map:   0%|          | 0/1778797 [00:00<?, ? examples/s]

Map:   0%|          | 0/96255 [00:00<?, ? examples/s]

Map:   0%|          | 0/96862 [00:00<?, ? examples/s]

### Examine class distribution
After removing duplicate comments, approximately 8% of observations are classified as toxic across the training, validation, and testing sets. This indicates a significant class imbalance, with non-toxic comments accounting for approximately 92% of the dataset.

We will preserve this distribution to maintain the original characteristics of the data. During model evaluation, we will consider precision, recall, F1-score, and PR-AUC in addition to accuracy, since a model could achieve high accuracy by disproportionately predicting the majority class.

In [21]:
# examine class distribution
for split in SPLITS:
    labels = np.array(ds[split][LABEL_COLUMN])

    total = len(labels)
    toxic = np.sum(labels == 1)
    non_toxic = np.sum(labels == 0)

    print(f"\n--- {split.upper()} ---")
    print("Total comments:", total)
    print("Toxic comments:", toxic)
    print("Non-toxic comments:", non_toxic)
    print("Percent toxic:", round(toxic / total * 100, 2))


--- TRAIN ---
Total comments: 1778797
Toxic comments: 142298
Non-toxic comments: 1636499
Percent toxic: 8.0

--- VALIDATION ---
Total comments: 96255
Toxic comments: 7585
Non-toxic comments: 88670
Percent toxic: 7.88

--- TEST ---
Total comments: 96862
Toxic comments: 7739
Non-toxic comments: 89123
Percent toxic: 7.99


### Verify data processing and cleaning

In [23]:
# final preprocessing verification
for split in SPLITS:
    data = ds[split]

    # verify binary labels
    labels = np.array(data[LABEL_COLUMN])
    scores = np.array(data[TARGET_COLUMN])

    assert np.isin(labels, [0, 1]).all()

    assert np.array_equal(
        labels,
        (scores >= TOXICITY_THRESHOLD).astype(int)
    )

    # verify unique comments
    assert len(set(data[TEXT_COLUMN])) == len(data)

    print(f"{split}: all checks passed")

# verify no overlap between splits
assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("\nData preparation completed successfully")

train: all checks passed
validation: all checks passed
test: all checks passed

Data preparation completed successfully


Save the processed dataset

In [25]:
ds.save_to_disk("../data/processed/civil_comments")

Saving the dataset (0/2 shards):   0%|          | 0/1778797 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/96255 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/96862 [00:00<?, ? examples/s]